# 🚀 E-Commerce Delivery Delay - Cloud Training Notebook

### **Instructions:**
1. Ensure you have selected the **Google Colab** kernel in the top right.
2. **Run the first cell** to upload your `features.csv` file from your local project.

In [ ]:
# --- 1. UPLOAD DATA ---
import ipywidgets as widgets
from IPython.display import display
import os

uploader = widgets.FileUpload(
    accept='.csv', 
    multiple=False,
    description='Select features.csv'
)
output = widgets.Output()

def on_upload_change(change):
    with output:
        output.clear_output()
        if not uploader.value: return
        # Handle ipywidgets v8 format (list of dicts)
        uploaded_file = uploader.value[0]
        filename = uploaded_file['name']
        content = uploaded_file['content']
        with open("features.csv", "wb") as f:
            f.write(content)
        print(f"✅ SUCCESS: Saved as 'features.csv' ({len(content)/1024/1024:.2f} MB)")

uploader.observe(on_upload_change, names='value')
display(uploader, output)

In [ ]:
# --- 2. INSTALL LIBRARIES ---
!pip install catboost lightgbm optuna pandas loguru

In [ ]:
import pandas as pd
import numpy as np
import optuna
import json
import os
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

# Settings
DATA_PATH = "features.csv"
N_TRIALS = 50
N_SPLITS = 3
TARGET = 'is_late'

# Load and Prep
if not os.path.exists(DATA_PATH):
    print("❌ ERROR: features.csv not found. Please upload it first!")
else:
    df = pd.read_csv(DATA_PATH)
    X = df.drop(columns=['order_id', TARGET])
    y = df[TARGET].astype(int)

    cat_cols = ['customer_state', 'seller_state', 'product_category', 
                'primary_payment_type', 'purchase_month', 'purchase_day_of_week', 'purchase_hour']
    for col in cat_cols:
        if col in X.columns:
            X[col] = X[col].fillna("UNKNOWN").astype(str)

    print(f"Loaded {len(df)} rows. Ready to optimize!")

In [ ]:
# --- 3. CATBOOST GPU TUNING ---
def cb_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 400, 1500),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 10),
        "task_type": "GPU",
        "auto_class_weights": "Balanced",
        "verbose": False
    }
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, cat_features=cat_cols)
        y_prob = model.predict_proba(X_val)[:, 1]
        scores.append(average_precision_score(y_val, y_prob))
    return np.mean(scores)

cb_study = optuna.create_study(direction="maximize")
cb_study.optimize(cb_objective, n_trials=N_TRIALS)
print(f"Best CatBoost PR-AUC: {cb_study.best_value}")

In [ ]:
# --- 4. LIGHTGBM GPU TUNING ---
X_lgb = X.copy()
for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype('category')

def lgb_objective(trial):
    params = {
        "objective": "binary",
        "device": "gpu",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 400, 1500),
        "num_leaves": trial.suggest_int("num_leaves", 31, 256),
        "is_unbalance": True,
        "verbose": -1
    }
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X_lgb, y):
        X_tr, X_val = X_lgb.iloc[tr_idx], X_lgb.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_val)[:, 1]
        scores.append(average_precision_score(y_val, y_prob))
    return np.mean(scores)

lgb_study = optuna.create_study(direction="maximize")
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS)
print(f"Best LightGBM PR-AUC: {lgb_study.best_value}")

In [ ]:
# --- 5. FINAL EXPORT ---
cb_final = CatBoostClassifier(**cb_study.best_params, task_type="GPU", auto_class_weights="Balanced", verbose=False)
cb_final.fit(X, y, cat_features=cat_cols)
cb_final.save_model("catboost_tuned.cbm")
with open("best_catboost_params.json", "w") as f: json.dump(cb_study.best_params, f)

lgb_final = lgb.LGBMClassifier(**lgb_study.best_params, device="gpu", is_unbalance=True)
lgb_final.fit(X_lgb, y)
lgb_final.booster_.save_model("lightgbm_tuned.txt")
with open("best_lightgbm_params.json", "w") as f: json.dump(lgb_study.best_params, f)

print("✅ All models saved. They will be downloaded automatically.")
from google.colab import files
!zip cloud_results.zip *.cbm *.txt *.json
files.download("cloud_results.zip")